# Double Query Rewriting Investigation

**Goal:** Check if the orchestrator LLM already rewrites the user query when calling the `search_handbook` tool, making the explicit `query_rewriting.py` step redundant.

**How it works:**
1. The orchestrator LLM receives the user question + conversation history and decides to call `search_handbook(query="...")` — this query is already a rewrite of the original question.
2. Then `query_rewriting.py` rewrites the query *again* for a second retrieval pass.

**What we log:**
- `original_question` — what the user typed
- `tool_query` — what the orchestrator LLM sent to the search tool (implicit rewrite)
- `explicit_rewrite` — what `query_rewriting.py` produces (explicit rewrite)

If `tool_query` and `explicit_rewrite` are semantically similar, the explicit rewriting step is redundant and adds unnecessary latency + cost.

## 1. Imports & Setup

In [1]:
import json
import os
import sys
import time
from datetime import date
from pathlib import Path

from dotenv import load_dotenv
from litellm import completion
from openai import OpenAI
import pandas as pd

# --- Path setup ---
# We want to import from the BACKEND code (not experiments/utils) since
# we're testing what the production pipeline actually does.
_cwd = Path.cwd()
_candidates = [_cwd, _cwd.parent, _cwd.parent.parent]
BACKEND_PATH = None
for root in _candidates:
    p = root / "backend"
    if p.exists():
        BACKEND_PATH = p
        break
if BACKEND_PATH is None:
    BACKEND_PATH = _cwd / "backend"

SRC_PATH = BACKEND_PATH / "src"

# Clear any cached utils module from experiments path
if "utils" in sys.modules:
    del sys.modules["utils"]
if "utils.prompts" in sys.modules:
    del sys.modules["utils.prompts"]

# Backend paths first — so utils resolves to backend/utils, not experiments/utils
for p in [str(BACKEND_PATH), str(SRC_PATH), str(SRC_PATH / "rag")]:
    if p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

# Load env
env_path = BACKEND_PATH / ".env"
if env_path.exists():
    load_dotenv(env_path, override=True)
    print(f"[OK] Loaded environment from {env_path}")
else:
    load_dotenv(override=True)
    print("[WARNING] Backend .env not found")

from utils.prompts import TOOL_DECISION_SYSTEM_PROMPT, REWRITE_QUERY_SYSTEM_PROMPT
from query_rewriting import rewrite_query

# Verify we loaded from backend, not experiments
import utils.prompts as _prompts_mod
print(f"[OK] Loaded prompts from: {_prompts_mod.__file__}")
assert "backend" in str(_prompts_mod.__file__), "ERROR: loaded prompts from experiments, not backend!"
print("[OK] All imports loaded from backend")

[OK] Loaded environment from c:\Users\smvan\repos\madetech-rag-assistant\backend\.env
[OK] Loaded prompts from: c:\Users\smvan\repos\madetech-rag-assistant\backend\utils\prompts.py
[OK] All imports loaded from backend


## 2. Configuration

In [2]:
MODEL = "groq/openai/gpt-oss-20b"

# Tool definition (same as pipeline.py)
SEARCH_HANDBOOK_TOOL = {
    "type": "function",
    "function": {
        "name": "search_handbook",
        "description": (
            "Search the Made Tech handbook for relevant information. "
            "Use this whenever the user asks about company policies, processes, "
            "benefits, roles, or any subject that requires handbook knowledge."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": (
                        "A concise search query that captures the core of what "
                        "the user wants to know."
                    ),
                }
            },
            "required": ["query"],
        },
    },
}

# Test scenarios: pairs of (conversation_history, follow_up_question)
# These simulate real multi-turn conversations where rewriting matters most
TEST_CASES = [
    {
        "label": "Follow-up about parental leave details",
        "history": [
            {"role": "user", "content": "What benefits does Made Tech offer?"},
            {"role": "assistant", "content": "Made Tech offers a range of benefits including pension scheme, private medical insurance, cycle to work scheme, flexible working, 38 days holiday per year, and more."},
        ],
        "question": "What about parental leave?",
    },
    {
        "label": "Follow-up about specific role after general question",
        "history": [
            {"role": "user", "content": "What roles exist at Made Tech?"},
            {"role": "assistant", "content": "Made Tech has over 50 documented roles across Delivery Management, Product/Design, Software Engineering, Data, Security, and more."},
        ],
        "question": "Tell me more about the lead one",
    },
    {
        "label": "Pronoun reference to previous topic",
        "history": [
            {"role": "user", "content": "How does the pension scheme work?"},
            {"role": "assistant", "content": "Made Tech offers a pension scheme through Scottish Widows with employer matching between 4-9% based on SFIA level."},
        ],
        "question": "How much do they match for senior engineers?",
    },
    {
        "label": "Implicit context from conversation",
        "history": [
            {"role": "user", "content": "What is the security policy?"},
            {"role": "assistant", "content": "Made Tech has comprehensive security policies covering passwords, BYOD, data protection, and device management."},
        ],
        "question": "And what about working from abroad?",
    },
    {
        "label": "Direct standalone question (no rewriting needed)",
        "history": [],
        "question": "What is the cycle to work scheme?",
    },
    {
        "label": "Another standalone question",
        "history": [],
        "question": "How many days of holiday do employees get?",
    },
    {
        "label": "Vague follow-up",
        "history": [
            {"role": "user", "content": "Tell me about onboarding"},
            {"role": "assistant", "content": "Onboarding at Made Tech includes a pre-start phase, first day remote call, and a group onboarding week covering 13 topics."},
        ],
        "question": "What happens in the first week?",
    },
    {
        "label": "Topic switch mid-conversation",
        "history": [
            {"role": "user", "content": "What are the delivery standards?"},
            {"role": "assistant", "content": "Made Tech has 11 delivery standards including daily standups, bi-weekly retros, and continuous delivery."},
        ],
        "question": "Actually, can you tell me about the dress code?",
    },
]

print(f"[OK] {len(TEST_CASES)} test cases configured")

[OK] 8 test cases configured


## 3. Run comparison

For each test case:
1. **Orchestrator tool call** — send question + history to the LLM with the tool definition, capture the `query` argument it generates
2. **Explicit rewrite** — call `rewrite_query()` with the same question + history
3. Compare both outputs side by side

In [3]:
def get_tool_query(question: str, history: list[dict], model: str) -> tuple[str | None, bool]:
    """
    Call the orchestrator LLM with the tool definition and return the query it generates.
    Returns (tool_query, used_tool). If the LLM answers directly without calling the tool,
    returns (None, False).
    """
    system_prompt = TOOL_DECISION_SYSTEM_PROMPT.format(today=date.today().strftime("%B %d, %Y"))
    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": question}]
    )
    response = completion(
        model=model,
        messages=messages,
        tools=[SEARCH_HANDBOOK_TOOL],
        tool_choice="auto",
    )
    msg = response.choices[0].message
    if not msg.tool_calls:
        return None, False
    tool_args = json.loads(msg.tool_calls[0].function.arguments)
    return tool_args.get("query", question), True


def get_explicit_rewrite(question: str, history: list[dict], model: str) -> str:
    """Call query_rewriting.py to get the explicit rewrite."""
    return rewrite_query(question, history, model)


# Run all test cases
results = []
for tc in TEST_CASES:
    label = tc["label"]
    question = tc["question"]
    history = tc["history"]
    
    print(f"\n{'='*60}")
    print(f"Test: {label}")
    print(f"Question: {question}")
    print(f"History: {len(history)} messages")
    
    # Get orchestrator tool query
    tool_query, used_tool = get_tool_query(question, history, MODEL)
    print(f"  Tool query:       {tool_query}")
    print(f"  Used tool:        {used_tool}")
    
    # Get explicit rewrite
    explicit = get_explicit_rewrite(question, history, MODEL)
    print(f"  Explicit rewrite: {explicit}")
    
    results.append({
        "label": label,
        "original_question": question,
        "has_history": len(history) > 0,
        "tool_used": used_tool,
        "tool_query": tool_query,
        "explicit_rewrite": explicit,
    })

print(f"\n{'='*60}")
print(f"Done: {len(results)} test cases completed")


Test: Follow-up about parental leave details
Question: What about parental leave?
History: 2 messages
  Tool query:       parental leave Made Tech handbook
  Used tool:        True
  Explicit rewrite: Made Tech parental leave policy

Test: Follow-up about specific role after general question
Question: Tell me more about the lead one
History: 2 messages
  Tool query:       None
  Used tool:        False
  Explicit rewrite: Lead role responsibilities at Made Tech

Test: Pronoun reference to previous topic
Question: How much do they match for senior engineers?
History: 2 messages
  Tool query:       pension scheme matching senior engineer SFIA level
  Used tool:        True
  Explicit rewrite: senior engineer pension employer match percentage

Test: Implicit context from conversation
Question: And what about working from abroad?
History: 2 messages
  Tool query:       working from abroad taking laptops abroad
  Used tool:        True
  Explicit rewrite: working from abroad policy

Test: 

## 4. Results comparison table

In [4]:
df = pd.DataFrame(results)

# Display full comparison
display_cols = ["label", "original_question", "tool_query", "explicit_rewrite"]
with pd.option_context("display.max_colwidth", 80, "display.max_rows", 20):
    display(df[display_cols])

,label,original_question,tool_query,explicit_rewrite
0,Follow-up about parental leave details,What about parental leave?,parental leave Made Tech handbook,Made Tech parental leave policy
1,Follow-up about specific role after general question,Tell me more about the lead one,NaN,Lead role responsibilities at Made Tech
2,Pronoun reference to previous topic,How much do they match for senior engineers?,pension scheme matching senior engineer SFIA level,senior engineer pension employer match percentage
3,Implicit context from conversation,And what about working from abroad?,working from abroad taking laptops abroad,working from abroad policy
4,Direct standalone question (no rewriting needed),What is the cycle to work scheme?,cycle to work scheme,Cycle to Work scheme policy\n\n
5,Another standalone question,How many days of holiday do employees get?,NaN,annual leave days employees
6,Vague follow-up,What happens in the first week?,first week onboarding Made Tech group week 13 topics,Made Tech onboarding first week schedule
7,Topic switch mid-conversation,"Actually, can you tell me about the dress code?",dress code,Made Tech dress code policy


## 5. Analysis

Key questions to answer:
1. Does the orchestrator already rewrite follow-up questions into standalone queries?
2. Does the explicit rewrite add meaningful information beyond what the tool query provides?
3. For standalone questions (no history), are both rewrites identical to the original?

In [5]:
# Summary stats
with_history = df[df["has_history"] == True]
without_history = df[df["has_history"] == False]
tool_skipped = df[df["tool_used"] == False]

print("=== Summary ===")
print(f"Total test cases:              {len(df)}")
print(f"  With conversation history:   {len(with_history)}")
print(f"  Without history (standalone): {len(without_history)}")
print(f"  Tool NOT called (answered directly): {len(tool_skipped)}")
print()

# For cases with history, check if the tool query already resolves the follow-up
print("=== Follow-up questions (with history) ===")
for _, row in with_history.iterrows():
    print(f"\n  [{row['label']}]")
    print(f"  Original:         {row['original_question']}")
    print(f"  Tool query:       {row['tool_query']}")
    print(f"  Explicit rewrite: {row['explicit_rewrite']}")
    
    # Simple heuristic: does the tool query already contain the key context?
    orig_words = set(row["original_question"].lower().split())
    tool_words = set(str(row["tool_query"]).lower().split()) if row["tool_query"] else set()
    explicit_words = set(row["explicit_rewrite"].lower().split())
    
    tool_added = tool_words - orig_words
    explicit_added = explicit_words - orig_words
    print(f"  Words added by tool query:       {tool_added or '(none)'}")
    print(f"  Words added by explicit rewrite: {explicit_added or '(none)'}")

print("\n=== Standalone questions (no history) ===")
for _, row in without_history.iterrows():
    print(f"\n  [{row['label']}]")
    print(f"  Original:         {row['original_question']}")
    print(f"  Tool query:       {row['tool_query']}")
    print(f"  Explicit rewrite: {row['explicit_rewrite']}")

=== Summary ===
Total test cases:              8
  With conversation history:   6
  Without history (standalone): 2
  Tool NOT called (answered directly): 2

=== Follow-up questions (with history) ===

  [Follow-up about parental leave details]
  Original:         What about parental leave?
  Tool query:       parental leave Made Tech handbook
  Explicit rewrite: Made Tech parental leave policy
  Words added by tool query:       {'tech', 'handbook', 'made', 'leave'}
  Words added by explicit rewrite: {'tech', 'policy', 'made', 'leave'}

  [Follow-up about specific role after general question]
  Original:         Tell me more about the lead one
  Tool query:       nan
  Explicit rewrite: Lead role responsibilities at Made Tech
  Words added by tool query:       {'nan'}
  Words added by explicit rewrite: {'at', 'made', 'responsibilities', 'role', 'tech'}

  [Pronoun reference to previous topic]
  Original:         How much do they match for senior engineers?
  Tool query:       pension s

## 6. Conclusion

After reviewing the results above, answer:

- **Is the explicit rewriting step redundant?** If the orchestrator's tool query already rewrites follow-ups into standalone queries with sufficient context, then `query_rewriting.py` adds latency and cost for no benefit.
- **Should we disable query rewriting?** This aligns with the experiment results from `03_experiment_rerank_rewrite` where basic RAG (no rewriting, no reranking) was the best configuration.
- **Edge cases:** Are there any test cases where the explicit rewrite captured context that the tool query missed?